# Chapter 5

In this chapter, we will look into loading data that has already been pre-processed in EEGLAB. We will also learn how to apply a vendor-supplied montage to set the sensor locations for the imported data.

For this chapter, we first need to download the data. Please download the **`eeg_montage_data`** folder from this [Pybrain 2020 MNE Google Drive](https://drive.google.com/drive/folders/1uxBHfSZd8r7JpFP-wnFBv_goBMAin6EJ).

## Libraries & Config

In [ ]:
import pathlib
import matplotlib

import mne

matplotlib.use("QtAgg")
mne.set_log_level("WARNING")

# Loading EEGLAB data

To load EEGLAB data that has already been preprocessed and epoched (saved as a `.set` file), we can use `mne.read_epochs_eeglab()`. This will directly load the data into an MNE Epochs object, preserving the event structure.

In [ ]:
eeglab_fname = pathlib.Path("eeg_montage_data") / "sample_eeglab_epochs.set"
epochs = mne.read_epochs_eeglab(eeglab_fname)

epochs

The loaded epochs contains 335 epochs balanced across 6 conditions - 100, 120, 140, 160, 180 and 200 – with frequencies - 56, 56, 56, 57, 55, 55 - respectively across the epochs, recorded from 64 EEG channels at a sampling rate of 200 Hz. Each epoch spans from -0.760 s to +2.560 s, and while the data includes sensor locations, it currently has no baseline correction and no high-pass filter applied (0.00 Hz), meaning baseline correction and filtering will likely be necessary preprocessing steps before ERP analysis.

In [ ]:
epochs.plot()

Now if we do `epochs.plot_sensors()` to see where all the sensors are located we will find them at random position and that is because we didn't import the **montage** which tells the locations of the sensors:

In [ ]:
epochs.plot_sensors()

## Loading vendor supplied montage

A montage provides the 3D coordinates for each sensor on the subject's head. We need to set this explicitly because raw data imported from other software often lacks spatial information. Without a montage, you cannot:
+ Create topographic maps ('heat maps').
+ Perform source localization.
+ Interpolate bad sensors using neighboring channels.

To fix this, we will call the `mne.channels.read_custom_montage()` func to load the manufacturer's template (in an ideal case, you will have montage with actual digitisation point of the sensors per participant):

In [ ]:
montage_fname = pathlib.Path("eeg_montage_data") / "achtiCHamp_64_channels_and_fiducials_Theta_Phi.txt"

# digatize the montageion from file
dig_monatge = mne.channels.read_custom_montage(
    fname=montage_fname
)

In [ ]:
dig_monatge.plot(sphere="auto") # sphere="auto" to center the head in the plot and scale it appropriately in the sense of 3D visualization

## Applying Monatage

To apply montage we can call the `.set_montage()` func of the *Epochs* instance:

In [ ]:
epochs.set_montage(dig_monatge)

In [ ]:
epochs.plot_sensors()

In summary, setting a montage is necessary because it links your channel names (like "Fz" or "Cz") to their physical 3D coordinates on the head, which *MNE* needs to perform any analysis that involves space. Without these coordinates, the software is "blind" to geometry, meaning you cannot create *topomaps* to visualize which brain areas are active, you cannot use *interpolation* to repair a bad sensor by averaging its neighbors (because MNE doesn't know which neighbors are close), and you cannot perform *source localization* to identify the deep brain structures generating the signals. <br />
In short, the montage turns your data from a simple list of voltage numbers into a meaningful 3D map of brain activity.


# The End & Have a great day!